# Week 9-1 · DMP-03 — OOP in Python (Part 1): EMH, Random Walks & Vectorized Back-testing
**Module:** Data Analysis & Modeling in Python · Instructor: Dr. Yves Hilpisch (The Python Quants).

This double lecture builds the *foundation* before OOP: **what we are up against** (the Efficient Market
Hypothesis), the **math** of returns and risk, and a first **vectorized back-test**. OOP and event-based
back-testing come in the second half — but you can't appreciate *why* OOP helps until you've felt the friction
of the vectorized approach.

> The lecture uses a 10-year EUR/USD end-of-day file (not shipped here), so we reproduce every idea on the
> shipped `eod_prices.csv` (NIFTY index, 962 daily bars 2017→2020). The concepts are identical.

## 1. The villain: Efficient Market Hypothesis (EMH)
Algorithmic trading = trying to **beat the market**. The EMH is the benchmark we run up against, and it has
*a ton* of empirical support. Three landmarks:
- **Fama (1960s):** prices behave ≈ a random walk; past price changes give no systematic forecasting power.
- **Jensen (1978):** a market is efficient w.r.t. an *information set* if no trading rule using only that
  information earns persistent, risk-adjusted excess profit.
- **Timmermann–Granger (2004):** efficiency depends on the information set **S**, the search technology **T**,
  and the model class **M** available at time *t* — so EMH is a **moving target** that changes with technology.

> If you firmly believe the EMH, you shouldn't do algorithmic trading at all. We treat it as a *moving
> benchmark* and try to find exploitable structure.

## 2. The random-walk model
$$S_{t+1} = S_t + \varepsilon_{t+1}, \qquad \mathbb{E}[\varepsilon_{t+1}\mid \mathcal{F}_t]=0$$
The best least-squares forecast of tomorrow's price given everything known today is simply **today's price**
$\hat S_{t+1}=S_t$. There is no predictable alpha from past prices alone. Let's simulate a few paths.

In [ ]:
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
steps, paths = 500, 5
shocks = rng.normal(0, 1, size=(steps, paths))
walks = 100 + np.cumsum(shocks, axis=0)        # S_t = S_0 + sum of white noise
print("simulated", paths, "random walks of", steps, "steps")
print("final levels:", np.round(walks[-1], 2))

## 3. Testing for predictability — prices vs returns
The key distinction in all of quant finance: **prices** vs **returns**. Prices are *trivially* autocorrelated
(today ≈ yesterday). The real test is on **returns**: in an efficient market their autocorrelation should sit
inside a statistical noise band.

In [ ]:
px = pd.read_csv("eod_prices.csv", index_col="Date", parse_dates=True)["Close"]
# price autocorrelation at lag 1 (shift by one day)
price_ac1 = px.corr(px.shift(1))
rets = np.log(px / px.shift(1)).dropna()        # log returns
ret_ac1 = rets.corr(rets.shift(1))
print(f"price  autocorr (lag 1): {price_ac1:.4f}   <- ~0.99, the random-walk result")
print(f"return autocorr (lag 1): {ret_ac1:.4f}   <- near zero, no usable price-based forecast")

### The autocorrelation function (ACF) and the noise band
We compute the ACF of returns out to several lags. The 95% confidence band is roughly $\pm 1.96/\sqrt{N}$.
Bars inside the band are statistical noise (efficient); persistent bars outside suggest **exploitable
structure**. We also build a *synthetic* AR(1) series with $\rho=0.30$ to show what inefficiency looks like.

In [ ]:
def acf(x, nlags=15):
    x = np.asarray(x); x = x - x.mean()
    denom = np.sum(x*x)
    return np.array([1.0]+[np.sum(x[k:]*x[:-k])/denom for k in range(1, nlags+1)])

N = len(rets)
band = 1.96/np.sqrt(N)
real_acf = acf(rets, 15)
n_out = int(np.sum(np.abs(real_acf[1:]) > band))

# synthetic AR(1): r_t = rho*r_{t-1} + noise   (rho>0 => exploitable structure)
rho = 0.30
e = rng.normal(0, 1, 4000); ar = np.zeros_like(e)
for t in range(1, len(e)):
    ar[t] = rho*ar[t-1] + e[t]
ar_acf = acf(ar, 15)

print(f"95% band = +/-{band:.4f}")
print("NIFTY return ACF lags 1-5:", np.round(real_acf[1:6], 4))
print(f"   -> {n_out} of 15 lags nudge past the band: small, scattered, no persistent sign (noisy but ~efficient)")
print(f"AR(1) rho={rho} ACF lags 1-5:", np.round(ar_acf[1:6], 4), "-> lag-1 clearly outside (exploitable)")

## 4. Returns & compounding
**Simple** returns compound *multiplicatively* (cumulative product of $1+R$); **log** returns add up
(cumulative sum). Log returns are analytically convenient and numerically stable over long / high-frequency
horizons.

In [ ]:
simple = px.pct_change().dropna()
logr   = np.log(px/px.shift(1)).dropna()
gross_simple = (1 + simple).prod() - 1     # multiplicative
gross_log    = logr.sum()                  # additive
print(f"cumulative SIMPLE return (cumprod): {gross_simple*100:.2f}%")
print(f"cumulative LOG return    (cumsum) : {gross_log*100:.2f}%  (= ln of price ratio)")
print(f"check exp(log) -> simple: {(np.exp(gross_log)-1)*100:.2f}%")

## 5. Risk metrics — volatility, Sharpe, drawdown
End-of-day data → annualize with **252** trading days: $\sigma_{ann}=\sigma_{daily}\sqrt{252}$,
$\text{Sharpe}=\mu_{ann}/\sigma_{ann}$. Then the **maximum drawdown** — the worst peak-to-trough fall.

In [ ]:
mu_d, sd_d = logr.mean(), logr.std()
mu_a, sd_a = mu_d*252, sd_d*np.sqrt(252)
sharpe = mu_a/sd_a
equity = (1+simple).cumprod()
roll_max = equity.cummax()
drawdown = equity/roll_max - 1
max_dd = drawdown.min()
print(f"annualized return     : {mu_a*100:6.2f}%")
print(f"annualized volatility : {sd_a*100:6.2f}%")
print(f"Sharpe ratio          : {sharpe:6.2f}")
print(f"maximum drawdown      : {max_dd*100:6.2f}%")

### The curse of asymmetry
Recovering from a loss needs a **larger** percentage gain than the loss itself. A 20% drawdown needs +25%; a
50% drawdown needs +100%. Volatility alone hides this — which is why drawdown matters so much.

In [ ]:
for loss in [0.05, 0.10, 0.20, 0.50, 0.90]:
    gain = loss/(1-loss)
    print(f"drawdown {loss*100:4.0f}%  ->  needs +{gain*100:6.1f}% to break even")

## 6. Vectorized back-testing — an OLS auto-regressive strategy
**Vectorized** = deterministic transformations on whole arrays (no `for` loop over bars). We build **7 lagged
return features**, fit an **OLS** linear model (alpha + a beta *vector*), forecast the next return, and take a
**long/short** position from the sign of the forecast. Strategy return = position × next return.

> **Look-ahead (foresight) bias** is the cardinal sin: the position for day *t* must use only information up to
> day *t-1*. We shift the position by one bar.

In [ ]:
lags = 7
df = pd.DataFrame({"r": logr})
cols = []
for k in range(1, lags+1):
    df[f"l{k}"] = df["r"].shift(k)        # lag features
    cols.append(f"l{k}")
df = df.dropna()

X = np.column_stack([np.ones(len(df))] + [df[c].values for c in cols])  # constant + 7 lags
y = df["r"].values
beta, *_ = np.linalg.lstsq(X, y, rcond=None)   # OLS: alpha + beta vector
print("OLS coefficients (alpha + 7 betas):")
print(np.round(beta, 5))

In [ ]:
df["pred"] = X @ beta                       # in-sample forecast of the return
df["pos"]  = np.sign(df["pred"]).shift(1)   # long/short on sign, shifted -> no look-ahead
df = df.dropna()
df["bh"]    = df["r"]                        # buy & hold (log) return
df["strat"] = df["pos"] * df["r"]           # strategy (log) return
bh_tot    = np.exp(df["bh"].sum())   - 1
strat_tot = np.exp(df["strat"].sum())- 1
print(f"positions: long {int((df.pos==1).sum())} / short {int((df.pos==-1).sum())}")
print(f"Buy & Hold total : {bh_tot*100:6.2f}%")
print(f"OLS strategy total: {strat_tot*100:6.2f}%")

### Equity curves — strategy vs buy & hold

In [ ]:
fig, ax = plt.subplots(figsize=(8,3.6))
ax.plot(df.index, df["bh"].cumsum().apply(np.exp),    color="#64748b", lw=1.2, ls="--",
        label=f"Buy & Hold ({bh_tot*100:.1f}%)")
ax.plot(df.index, df["strat"].cumsum().apply(np.exp), color="#0f766e", lw=1.4,
        label=f"OLS AR(7) strategy ({strat_tot*100:.1f}%)")
ax.axhline(1, color="gray", lw=0.6)
ax.set_title("Vectorized OLS auto-regressive back-test (NIFTY daily)")
ax.set_ylabel("Growth of 1"); ax.legend(); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig("preview.png", dpi=110)
print("equity curves drawn")

### Honest reading of the result
Striking: even as an **in-sample** fit with **no transaction costs** — the setup that should *flatter* a
strategy — the OLS sign rule **loses money** (≈ -47%) while buy-and-hold gains ≈ 54%. The EMH warned us returns
are barely predictable, and the autocorrelation test agreed (ACF scattered near zero with no persistent sign).
This is a textbook demonstration of *why* a strategy must clear the **quality checklist** before any capital is
risked — an arbitrary signal, even fitted to the data, has no real edge.

## 7. The strategy quality checklist (a pilot's pre-flight)
| Gate | Question |
|---|---|
| Statistical edge | Do returns/forecasts differ from noise? (the ACF test) |
| Economic edge | Profitable **after** financing & transaction costs? |
| Persistence / robustness | Works **out-of-sample**, not just on one tiny set? |
| Implementability | Are there liquid, tradable instruments? (an index isn't tradable directly) |
| Benchmark & capital cost | Beats the benchmark and your cost of capital? |
| Risk profile | Drawdowns, volatility, tail behaviour acceptable? |

Hilpisch's retail-friendly playground: **FX** — highly liquid, tight spreads, symmetric long/short, clean REST
APIs, and mean-reverting tendencies around macro regimes.

## Key takeaways
- **EMH is the benchmark**, not a law — a *moving target* (Timmermann–Granger: depends on info set S, search tech T, model class M). If you believe it fully, don't trade algorithmically.
- **Random walk:** best forecast of tomorrow's price is today's price; no alpha from past prices alone.
- **Prices vs returns:** prices autocorrelate ~0.99; **returns** are the real test — here ACF scatters near zero (no persistent structure). A synthetic AR(1) with ρ=0.30 shows what *exploitable* structure actually looks like.
- **Compounding:** simple returns multiply (cumprod), log returns add (cumsum); log is numerically stable.
- **Risk:** annualize with √252; Sharpe = μ/σ; watch **max drawdown** and the **curse of asymmetry** (50% loss needs +100% to recover).
- **Vectorized back-test:** whole-array transforms, no loop; build lag features → OLS → sign → **shift to avoid look-ahead** → strategy = position × return. Even in-sample with no costs the OLS rule **lost** (≈ -47% vs +54% buy-and-hold) — a clean EMH lesson.
- Judge every strategy against the **quality checklist** (statistical + economic edge, robustness, implementability, benchmark, risk) — this motivates the move to OOP & event-based back-testing in Part 2.

# Additive validation layer

The cells below are appended only in this validated copy. The original DMP-03 practice notebook remains unchanged.

In [ ]:
import pandas as pd
from pathlib import Path
base = Path('.')
files = ['dmp03_html_audit.csv', 'dmp03_source_inventory.csv', 'dmp03_notebook_execution.csv', 'dmp03_emh_autocorr_checks.csv', 'dmp03_return_risk_metrics.csv', 'dmp03_ols_backtest_metrics.csv', 'dmp03_validation_controls.csv']
data = {f: pd.read_csv(base / f) for f in files}
{k: v.shape for k, v in data.items()}

## 1. Source and notebook audit

In [ ]:
print(data['dmp03_source_inventory.csv'].to_string(index=False))
print(data['dmp03_notebook_execution.csv'].to_string(index=False))
nb = data['dmp03_notebook_execution.csv'].set_index('metric')
assert int(nb.loc['original_execution_errors','value']) == 0

## 2. EMH and autocorrelation checks

In [ ]:
print(data['dmp03_emh_autocorr_checks.csv'].to_string(index=False))
acf = data['dmp03_emh_autocorr_checks.csv'].set_index('check')
assert int(acf.loc['nifty_lags_outside_band','value']) == 8
assert float(acf.loc['price_autocorr_lag1','value']) > 0.99

## 3. Return, risk, and OLS backtest metrics

In [ ]:
print(data['dmp03_return_risk_metrics.csv'].to_string(index=False))
print(data['dmp03_ols_backtest_metrics.csv'].to_string(index=False))
ols = data['dmp03_ols_backtest_metrics.csv'].set_index('metric')
assert int(ols.loc['ols_rows','value']) == 953
assert float(ols.loc['ols_strategy_total_pct','value']) < 0

## 4. Validation controls

In [ ]:
print(data['dmp03_validation_controls.csv'].to_string(index=False))
assert data['dmp03_validation_controls.csv'].shape[0] >= 6